# Caracterización del clúster pequeño (R2-11)

Notebook independiente para responder al Revisor 2 (comentario 11): localiza el clúster más pequeño (~1-1.5% de la muestra), intenta separar sus 'dos islas' en la representación UMAP (DBSCAN, con K-Means k=2 de respaldo), y caracteriza cada subgrupo.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/cluster_pequeno/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/cluster_pequeno'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Partición 'publicada' de referencia (K=8) y localizar el clúster más pequeño

In [ ]:
# Partición base K=8 (misma metodología del manuscrito: UMAP fit sobre
# 80,000 filas, transform sobre el resto; K-Means K=8), para tener una
# referencia 'publicada' propia sin depender de otros notebooks.
SEED_BASE = 100
K = 8
N_FIT = 80_000
N_EVAL = 50_000

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

rng_fit = np.random.default_rng(seed=SEED_BASE)
idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
rng_eval = np.random.default_rng(seed=0)
idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)

log("Ajustando UMAP base (K=8, seed=42)...")
reducer_base = umap_cpu.UMAP(n_components=2, random_state=SEED_BASE, n_neighbors=10,
                              low_memory=True, n_jobs=-1)
reducer_base.fit(X_full[idx_fit])
emb_eval_base = reducer_base.transform(X_full[idx_eval])
km_base = MiniBatchKMeans(n_clusters=K, random_state=SEED_BASE, n_init="auto", batch_size=10_000)
labels_pub = km_base.fit_predict(emb_eval_base)
log(f"Partición base lista. Tamaños: {np.bincount(labels_pub)}")

In [ ]:
sizes = np.bincount(labels_pub)
cluster_pequeno = int(np.argmin(sizes))
print(f"Tamaños de clúster: {sizes}")
print(f"Clúster más pequeño = {cluster_pequeno} (n={sizes[cluster_pequeno]}, "
      f"{sizes[cluster_pequeno]/len(labels_pub)*100:.2f}% del conjunto de evaluación)")

## 4. Separar las 'dos islas' (DBSCAN sobre las coordenadas UMAP del clúster)

In [ ]:
from sklearn.cluster import DBSCAN

mask_c = labels_pub == cluster_pequeno
emb_c = emb_eval_base[mask_c]
idx_eval_c = idx_eval[mask_c]

db = DBSCAN(eps=0.5, min_samples=5).fit(emb_c)
islas = db.labels_
vals, counts = np.unique(islas, return_counts=True)
print(f"Islas encontradas (DBSCAN): {dict(zip(vals.tolist(), counts.tolist()))}")

if len(vals[vals != -1]) != 2:
    print("DBSCAN no dio exactamente 2 islas -> usando KMeans(k=2) de respaldo")
    km2 = MiniBatchKMeans(n_clusters=2, random_state=42, n_init="auto")
    islas = km2.fit_predict(emb_c)

## 5. Caracterizar cada subgrupo (puntaje, instituciones, internet, becas, departamento)

In [ ]:
COLS_ORD = ['FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
            'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
COLS_PUNT = ['MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
             'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT', 'PUNT_GLOBAL']

df_c = df_filtrado_full.iloc[idx_eval_c].copy()
df_c['isla'] = islas
df_global = df_filtrado_full

perfil = {}
for isla_id in sorted(df_c['isla'].unique()):
    if isla_id == -1:
        continue
    sub = df_c[df_c['isla'] == isla_id]
    perfil[int(isla_id)] = {
        'n': int(len(sub)), 'pct_del_cluster': float(len(sub) / len(df_c) * 100),
        'punt_global_medio': float(sub['PUNT_GLOBAL'].mean()),
        'n_instituciones_distintas': int(sub['INST_COD_INSTITUCION'].nunique()),
        'internet_si_pct': float((sub['FAMI_TIENEINTERNET'] == 'Si').mean() * 100),
        'beca_si_pct': float((sub['ESTU_PAGOMATRICULABECA'] == 'Si').mean() * 100),
    }
    print(f"Isla {isla_id}: n={len(sub)} ({len(sub)/len(df_c)*100:.1f}% del clúster)  "
          f"PUNT_GLOBAL medio={sub['PUNT_GLOBAL'].mean():.1f} "
          f"(ref global={df_global['PUNT_GLOBAL'].mean():.1f})  "
          f"instituciones distintas={sub['INST_COD_INSTITUCION'].nunique()}  "
          f"internet={(sub['FAMI_TIENEINTERNET']=='Si').mean()*100:.1f}%")

json.dump({'cluster_pequeno_id': cluster_pequeno, 'n_cluster': int(sizes[cluster_pequeno]),
           'perfil_por_isla': perfil}, open(os.path.join(OUT_DIR, 'resultados.json'), 'w'), indent=2)
print("\nNota: la reproducibilidad de este clúster a través de múltiples corridas "
      "(15 repeticiones) se calcula en el notebook estabilidad_pipeline_colab.ipynb.")

## 6. Figura (Supplementary Figure S9)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=150)

mask_otros = labels_pub != cluster_pequeno
axes[0].scatter(emb_eval_base[mask_otros, 0], emb_eval_base[mask_otros, 1],
                 s=3, color='lightgray', alpha=0.5, label='Otros cl\u00fasteres')
axes[0].scatter(emb_c[:, 0], emb_c[:, 1], s=6, color='#B33F3F',
                 label=f'Cl\u00faster peque\u00f1o (n={sizes[cluster_pequeno]}, '
                       f'{sizes[cluster_pequeno] / len(labels_pub) * 100:.2f}%)')
axes[0].set_title('Ubicaci\u00f3n del cl\u00faster peque\u00f1o\nen el embedding UMAP completo')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

colores_isla = ['#3B6FA0', '#E08030', '#4C9F70']
for i, isla_id in enumerate(sorted(set(islas) - {-1})):
    mask_i = islas == isla_id
    axes[1].scatter(emb_c[mask_i, 0], emb_c[mask_i, 1], s=12, color=colores_isla[i % len(colores_isla)],
                     label=f'Subgrupo {i} (n={mask_i.sum()})')
axes[1].set_title('Vista ampliada del cl\u00faster peque\u00f1o:\n\u00bfdos islas separadas?')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_cluster_pequeno.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S9)")
